In [ ]:
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

COMP_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
TRAIN_CSV = COMP_DIR / 'train.csv'
REPO_URL = 'https://github.com/MananJain39/RSNA-knee-abnormality.git'
SNAPSHOT_REF = 'mananjain39/rsna-knee-repo-snapshot'
REPO_DIR = Path('/kaggle/working/rsna-knee-abnormality')
OUTPUT_DIR = Path('/kaggle/working/rsna-knee-processed-sample200')
TOTAL_CAP = 200
SEED = 42
IMAGE_SIZE = 336
CROP_EXTENT_MM = 130.0
KERNEL_VERSION = 'sample200-v7'

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f'Competition train.csv not found: {TRAIN_CSV}')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
# Only reset OUTPUT_DIR when no manifest.json is present. A completed
# prior run (manifest exists) means the slow DICOM preprocessing can be
# reused on re-runs that only need to retry the upload step.
manifest_path = OUTPUT_DIR / 'manifest.json'
if OUTPUT_DIR.exists() and not manifest_path.exists():
    shutil.rmtree(OUTPUT_DIR)

snapshot_zip_dir = Path('/kaggle/working/snapshot_dl')
snapshot_zip_dir.mkdir(parents=True, exist_ok=True)
dl = subprocess.run(
    ['kaggle', 'datasets', 'download', SNAPSHOT_REF,
     '-p', str(snapshot_zip_dir), '--unzip'],
    capture_output=True, text=True,
)
if dl.returncode != 0:
    raise RuntimeError(
        f'kaggle datasets download failed: rc={dl.returncode}\n'
        f'stdout: {dl.stdout}\nstderr: {dl.stderr}'
    )
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(snapshot_zip_dir, REPO_DIR)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'],
    cwd=REPO_DIR,
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.imaging.clustering import _syntax_family
from src.imaging.dicom_loader import get_series_metadata, load_series
from src.imaging.pipeline import preprocess_series
from src.imaging.slot_builder import SLOT_NAMES, build_slots_for_study

train = pd.read_csv(TRAIN_CSV)
gold_uids = sorted(train.loc[train['ACL'].notna(), 'StudyInstanceUID'].unique())
non_gold_pool = train.loc[~train['StudyInstanceUID'].isin(gold_uids), 'StudyInstanceUID'].drop_duplicates().tolist()
if len(gold_uids) != 58:
    raise ValueError(f'Expected 58 gold studies, found {len(gold_uids)}')

rng = np.random.default_rng(SEED)
n_non_gold = min(TOTAL_CAP - len(gold_uids), len(non_gold_pool))
non_gold_uids = sorted(rng.choice(non_gold_pool, size=n_non_gold, replace=False).tolist())
study_uids = sorted(set(gold_uids) | set(non_gold_uids))
if len(study_uids) != TOTAL_CAP:
    raise ValueError(f'Expected {TOTAL_CAP} selected studies, found {len(study_uids)}')

print(f'Selected {len(study_uids)} studies: {len(gold_uids)} gold + {len(non_gold_uids)} non-gold')


In [ ]:
TRAIN_DIR = COMP_DIR / 'train_series'
SERIES_CSV = COMP_DIR / 'train_series.csv'
if not SERIES_CSV.exists():
    raise FileNotFoundError(f'Competition train_series.csv not found: {SERIES_CSV}')

series_df = pd.read_csv(SERIES_CSV)
series_df = series_df[series_df['StudyInstanceUID'].isin(study_uids)].copy()
metadata_rows = []
failures = []


def _metadata_row(study_uid, series_uid, series_row, meta):
    return {
        'StudyInstanceUID': study_uid,
        'SeriesInstanceUID': series_uid,
        'pixel_spacing_mm': float(meta['pixel_spacing'][0]),
        'rows': int(meta['rows']),
        'cols': int(meta['columns']),
        'n_slices': int(meta['n_slices']),
        'slice_thickness_mm': float(meta['slice_thickness']),
        'fov_extent_mm': float(meta['pixel_spacing'][0]) * int(meta['rows']),
        'fluid_sensitive': int(series_row['Fluid_Sensitive']),
        'fat_suppression': int(series_row['Fat_Suppression']),
        'transfer_syntax_family': _syntax_family(meta['transfer_syntax']),
    }


for study_uid in tqdm(study_uids, desc='preprocess'):
    study_dir = OUTPUT_DIR / study_uid
    slots = build_slots_for_study(study_uid, series_df)
    saved_any = False

    for slot_name in SLOT_NAMES:
        slot = slots[slot_name]
        # Missing slots are expected: the 5-slot design has ~92.9% mean
        # fill rate per DECISION.md Decision 4. Skip silently; do not
        # record as a failure. Only real exceptions (corrupt DICOMs,
        # missing studies, unsupported transfer syntax) belong in
        # preprocessing_failures.csv.
        if slot is None:
            continue

        try:
            meta = get_series_metadata(study_uid, slot.series_uid, TRAIN_DIR)
            volume = load_series(study_uid, slot.series_uid, TRAIN_DIR)
            processed = preprocess_series(
                volume,
                meta,
                slot.anatomical_plane,
                image_size=IMAGE_SIZE,
                crop_extent_mm=CROP_EXTENT_MM,
            )
            if processed.dtype != np.float32:
                processed = processed.astype(np.float32)
            study_dir.mkdir(parents=True, exist_ok=True)
            np.save(study_dir / f'{slot_name}.npy', processed)

            series_row = series_df.loc[
                (series_df['StudyInstanceUID'] == study_uid)
                & (series_df['SeriesInstanceUID'] == slot.series_uid)
            ].iloc[0]
            metadata_rows.append(_metadata_row(study_uid, slot.series_uid, series_row, meta))
            saved_any = True
        except Exception as exc:
            failures.append({
                'StudyInstanceUID': study_uid,
                'slot': slot_name,
                'error': str(exc),
            })

    if not saved_any and study_dir.exists():
        shutil.rmtree(study_dir)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(metadata_rows).to_csv(OUTPUT_DIR / 'series_metadata.csv', index=False)
failure_columns = ['StudyInstanceUID', 'slot', 'error']
pd.DataFrame(failures, columns=failure_columns).to_csv(OUTPUT_DIR / 'preprocessing_failures.csv', index=False)

print(f'Preprocessed {len(study_uids)} studies and {len(metadata_rows)} series; failures={len(failures)}')


In [ ]:
try:
    repo_revision = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
        text=True,
    ).strip()
except Exception:
    repo_revision = 'snapshot-unknown'
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'kernel_version': KERNEL_VERSION,
    'repo_url': REPO_URL,
    'repo_revision': repo_revision,
    'n_studies': len(study_uids),
    'n_gold': len(gold_uids),
    'n_non_gold': len(non_gold_uids),
    'gold_uids': gold_uids,
    'study_uids': study_uids,
    'image_size': IMAGE_SIZE,
    'crop_extent_mm': CROP_EXTENT_MM,
    'n_metadata_rows': len(metadata_rows),
    'n_preprocessing_failures': len(failures),
    'preprocessing_failures': failures,
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

payload_bytes = sum(path.stat().st_size for path in OUTPUT_DIR.rglob('*') if path.is_file())
print(
    f'Payload ready: {manifest["n_studies"]} studies, '
    f'{payload_bytes / 1024**2:.1f} MiB, '
    f'{manifest["n_preprocessing_failures"]} failures'
)


In [ ]:
import zipfile

USERNAME = 'mananjain39'
SLUG = 'rsna-knee-processed-sample200'

# The kaggle CLI only uploads top-level files (subfolders are skipped
# or zipped per-folder), so ship the payload as ONE zip. Store it under
# /kaggle/tmp because payload (~13 GiB) + zip would exceed the 20 GiB
# /kaggle/working quota. ZIP_STORED: float32 npy compresses poorly.
UPLOAD_DIR = Path('/kaggle/tmp/upload')
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True)
ZIP_PATH = UPLOAD_DIR / f'{SLUG}.zip'
print(f'Zipping {OUTPUT_DIR} -> {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf:
    for p in sorted(OUTPUT_DIR.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR))
print(f'Zip ready: {ZIP_PATH.stat().st_size / 1024**2:.1f} MiB')

for name in ('manifest.json', 'series_metadata.csv', 'preprocessing_failures.csv'):
    shutil.copy2(OUTPUT_DIR / name, UPLOAD_DIR / name)
dataset_metadata = {
    'title': 'RSNA Knee Processed Sample 200',
    'id': f'{USERNAME}/{SLUG}',
    'isPrivate': True,
    'licenses': [{'name': 'CC0-1.0'}],
    'keywords': ['rsna', 'knee', 'mri', 'preprocessed'],
    'description': (
        'Private, deterministic 200-study RSNA knee sample: 58 gold studies '
        'plus 142 non-gold studies, processed into five-slot float32 MRI '
        'volumes. Single-file payload zip plus top-level metadata.'
    ),
}
(UPLOAD_DIR / 'dataset-metadata.json').write_text(
    json.dumps(dataset_metadata, indent=2),
    encoding='utf-8',
)

def _run(cmd, timeout=7200):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f'command {cmd} timed out after {timeout}s')


print(f'Uploading {UPLOAD_DIR} as new version of Kaggle dataset {USERNAME}/{SLUG}...')
# The dataset slug already exists (metadata-only v1), so `datasets create`
# would fail with a title conflict — and CLI 2.0.2 exits 0 on that error.
# Go straight to `datasets version` (NEXT_STEPS.md Phase A, Risk #1).
version = _run([
    'kaggle', 'datasets', 'version',
    '-p', str(UPLOAD_DIR),
    '-m', f'kernel {KERNEL_VERSION}',
])
print('version rc:', version.returncode)
print('stdout:', version.stdout)
print('stderr:', version.stderr)
combined = (version.stdout or '') + (version.stderr or '')
if version.returncode != 0 or 'error' in combined.lower():
    raise RuntimeError(
        f'kaggle datasets version failed: rc={version.returncode}\n'
        f'stdout: {version.stdout}\nstderr: {version.stderr}'
    )

print(f'Dataset version submitted: https://www.kaggle.com/datasets/{USERNAME}/{SLUG}')
